# Trabajo Practico 2 - Laboratorio de Datos - Verano 2026
- Prado Carlos Alberto
- Mia Modini
- Brian Tarqui

In [ ]:
# !pip install formulaic

In [ ]:
import pandas as pd
import numpy as np
import seaborn.objects as so
import seaborn as sns
from formulaic import Formula

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, r2_score, root_mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge

import keras
import tensorflow as tf

# Limpieza de datos

## Ejercicio 1
Leemos el archivo

In [ ]:
datos = pd.read_csv("usu_individual_T325.txt", sep = ";")
datos.head()

## Ejercicio 2
Eliminamos a todos los individuos que no compeltaron la encuesta(H15 = 1 significa "Entrevista indiviudal realizada")

In [ ]:
datos = datos[datos["H15"] == 1]
datos.head()

## Ejercicio 3
Seleccionamos las columnas con las que vamos a trabajar

In [ ]:
columnas = [
    "REGION", "MAS_500", "CH04", "CH03", "CH06", "CH07", "CH09", "CH10",
    "NIVEL_ED", "ESTADO", "CAT_OCUP", "CAT_INAC", "SECTOR", "PP02B",
    "PP02C1", "PP02C2", "PP02C3", "PP02C4", "PP02C5", "PP02C6", "PP02C7",
    "PP02C8", "PP02D", "PP02F", "PP02G", "PP02H", "PP02I", "PP03C",
    "PP03D", "PP3E_TOT", "PP3F_TOT", "PP03G", "PP03H", "PP04A", "PP04A1",
    "PP04B1", "PP04B2", "PP04B3_ANO", "PP04C", "PP03I", "PP03J", "PP03K",
    "INTENSI", "PP04G", "P47T", "EMPLEO"
]

datos = datos[columnas]
datos.head()

## Ejercicio 4
Reemplazamos datos faltantes por 0 en las columnas por preguntas que no corresponden como preguntar por el empleo a personas desocupadas

In [ ]:
columnas_1 = [
    "EMPLEO", "PP03C", "PP03D", "PP3E_TOT", "PP3F_TOT", "PP03G",
    "PP03H", "PP03I", "PP03J", "PP03K", "PP04B1", "PP04B2",
    "PP04B3_ANO", "PP04C", "PP04A", "PP04A1", "PP04G", "INTENSI", "SECTOR"
]

datos.loc[:, columnas_1] = datos.loc[:, columnas_1].fillna(0)
datos

## Ejercicio 5
Eliminamos las filas que no tienen valor en la columna de ingreso total("P47T")

In [ ]:
datos=datos[datos["P47T"] != 0]
datos

Tambien eliminamos las personas que tengan sueldo negativo

## Ejericio 6
Eliminamos filas con datos faltantes


In [ ]:
datos = datos.dropna()
datos

## Ejercicio 7
Convertir las variables categoricas seleccionadas a dummies

In [ ]:
variables_categoricas = [
    "ESTADO", "REGION", "MAS_500", "CAT_OCUP", "CAT_INAC", "PP02B", "PP02C1",
    "PP02C2", "PP02C3", "PP02C4", "PP02C5", "PP02C6", "PP02C7", "PP02C8",
    "PP02D", "PP02F", "PP02G", "PP02H", "PP02I", "PP03C", "PP03D", "PP03G",
    "PP03H", "PP03I", "PP03J", "PP03K", "INTENSI", "PP04A", "PP04A1", "PP04G"
]

datos_dummies = pd.get_dummies(datos, columns=variables_categoricas, drop_first=True)
datos_dummies

## Ejercicio 8
Se realizan las modficaciones propuestas
*   Reemplazar CH07 (estado civil) por una variable que indique si es soltero o no (para no usar tantas dummies).
*   Reemplazar CH09 por una variable que indique si sabe leer.
*   Eliminar a los individuos que no respondieron PP04C (cantidad de personas que trabajan en el lugar de trabajo), corresponden a codigo 99.
*   En nivel educativo, reemplazar 7 por 0.



In [ ]:
datos_dummies["CH07"] = (datos_dummies["CH07"] == 5).astype(int)
datos_dummies["CH09"] = (datos_dummies["CH09"] == 1).astype(int)
datos_dummies = datos_dummies[datos_dummies["PP04C"] != 99]
datos_dummies['NIVEL_ED'] = datos_dummies['NIVEL_ED'].replace(7, 0)


In [ ]:
# df_clean = pd.get_dummies(datos, columns=variables_categoricas, drop_first=True)
df_clean = datos_dummies.copy()

In [ ]:
df_clean

# Clustering

## Ejercicio 9

In [ ]:
df_clustering = pd.get_dummies(df_clean, columns=["SECTOR"], drop_first=True)

In [ ]:
df_clustering.head()

## Ejercicio 10
Calculamos las dos primeras componentes principales Z1 y Z2 del DataFrame df_clustering y realizamos un grafico de dispersion de Z1 vs Z2

In [ ]:
df_clustering_scaled = StandardScaler().set_output(transform="pandas").fit_transform(df_clustering)

pca = PCA(n_components=2)
componentes_principales = pca.fit_transform(df_clustering_scaled)

In [ ]:
(
    so.Plot(
        x=componentes_principales[:, 0],
        y=componentes_principales[:, 1],
    )
    .add(so.Dot(edgecolor="w", alpha=0.7))
).layout(size=(8,8))

A simple vista podemos observar una distincion en grupos. Hay varias formas de verlo:

- cuatro grupos separados por densidad
- un grupo con segunda componente aproximadamente 0 (horizontal) y otro grupo con primera componente negativa (vertical)

## Ejercicio 11

In [ ]:
# Pruebo kmeans con cuatro clusters
kmeans = KMeans(n_clusters=4, random_state=3, verbose=1)
kmeans.fit(df_clustering_scaled)

kmeans_labels = kmeans.labels_
labels = pd.Series(kmeans_labels).astype("category")

## Ejercicio 12
Realizamos el mismo grafico pero coloreando usando las etiquetas obtenidas de kmeans

In [ ]:
(
    so.Plot(x=componentes_principales[:, 0], y=componentes_principales[:, 1], color=labels)
    .add(so.Dot(edgecolor="w", alpha=0.7))
    .label(x="Z1", y="Z2")
).layout(size=(8,8))

Podemos observar que quedaron:

- un cluster con valor de Z2 mayor a 20
- un cluster con valor de Z1 negativo y Z2 menor a 5
- otros dos clusters con valor de Z2 cercano a 0, y Z1 mayor a 0

Vemos que el ultimo par de clusters no esta tan visiblemente separado como los otros

In [ ]:
# Pruebo dbscan

# d = DBSCAN(eps=1, min_samples=5)
# d.fit(df_clustering_scaled)

# dbscan_labels = d.labels_
# labels = pd.Series(dbscan_labels).astype("category")

# (
#     so.Plot(x=componentes_principales[:, 0], y=componentes_principales[:, 1], color=labels)
#     .add(so.Dot(edgecolor="w", alpha=0.7))
# ).layout(size=(12,8))

Al realizar la visualizacion usando DBSCAN, no fue posible encontrar clusters bien definidos.

El algoritmo de DBSCAN probablemente no sea la mejor opcion en este caso, ya que al haber muchas features puede perder sentido la nocion de distancia

## Ejercicio 13
Para identificar caracteristicas de los distintos clusters, armamos una tabla con los nombres de cada columna y sus respectivas componentes de las componentes principales.

De esta forma, podemos ver cuales columnas tienen mayor relevancia, ordenando por alguna de las componentes principales.

In [ ]:
direcciones = pd.DataFrame(
    pca.components_.T,
    columns=["Z1", "Z2"],
    index=[c for c in df_clustering.columns]
)

col_details = pd.Series({
    "PP04B1": "Si presta servicio doméstico en hogares particulares (1: casa de familia)",
    "EMPLEO": "1: formal, 2: informal, 3: ns/nr",
    "PP03C_1.0": "La semana pasada tenía un solo empleo",
    "PP02": "Relac con buscar trabajo (2 corresponde a NO)",
    "ESTADO_3": "Condicion de actividad: Inactivo",
    "CAT_INAC_1": "Categ inactividad: JUBILADO",
    "PP03": "Ocupados buscaron trabajo (2 es NO)",
    "ESTADO": "Actividad: 1 ocupado, 2 desocupado, 3 inactivo"
})


def get_description(col):
    for key, desc in col_details.items():
        if key in col:
            return desc
    return None

direcciones["detalles"] = direcciones.index.to_series().apply(get_description)

In [ ]:
# Al ordenar por Z1, queda primera la variable PP04B1, correspondiente a servicio domestico
direcciones.sort_values(by="Z1", ascending=False, inplace=True)
direcciones[:5]

In [ ]:
labels_pca = pd.Series(df_clustering["PP04B1"]).astype("category")
(
    so.Plot(
        x=componentes_principales[:, 0],
        y=componentes_principales[:, 1],
        color=labels_pca
    )
    .add(so.Dot(edgecolor="w", alpha=0.7))
    .label(x="Z1", y="Z2", title="Esta o no empleado")
).layout(size=(8,6))

In [ ]:
# Al ordenar por Z2, queda primera la variable PP02B_1, correspondiente a si durante
# los ultimos 30 dias estuvo buscando trabajo
direcciones.sort_values(by="Z2", ascending=False, inplace=True)
direcciones[:5]

In [ ]:
labels_pca = pd.Series(df_clustering["PP02B_1"]).astype("category")
(
    so.Plot(
        x=componentes_principales[:, 0],
        y=componentes_principales[:, 1],
        color=labels_pca
    )
    .add(so.Dot(edgecolor="w", alpha=0.7))
    .label(x="Z1", y="Z2", title="Desocupados no buscaron trabajo")
).layout(size=(8,6))

In [ ]:
# Al ordenar por Z1 y Z2, queda primera la variable ESTADO_3, correspondiente a inactivo
# Podemos ver que tambien esta la variable CAT_INAC_1, que corresponde a Jubilados
direcciones.sort_values(by=["Z1", "Z2"], ascending=True, inplace=True)
direcciones[:5]

In [ ]:
labels_pca = pd.Series(df_clustering["ESTADO_3"]).astype("category")
(
    so.Plot(
        x=componentes_principales[:, 0],
        y=componentes_principales[:, 1],
        color=labels_pca
    )
    .add(so.Dot(edgecolor="w", alpha=0.7))
    .label(x="Z1", y="Z2", title="Jubilados")
).layout(size=(8,6))

A partir del analisis, podemos ver algunos patrones:
- El cluster que figura a la derecha en el grafico, corresponde a personas que estan de alguna manera empleadas, especificamente son aquellos que prestan servicio domestico.
- En el segundo grafico podemos observar que hay una clara distincion entre personas que estuvieron buscando empleo en los ultimos 30 dias: el cluster que esta arriba corresponde a personas que no estuvieron buscando trabajo
- En el tercer grafico se observa un cluster que corresponde a personas inactivas, especificamente jubilados.

# Clasificacion

## Ejercicio 14
Generamos un DataFrame df_clasificacion que contenga solo los individuos con valores 1, 2 o 3 en la columna SECTOR

In [ ]:
df_clasificacion = df_clean.copy()

In [ ]:
df_clasificacion = df_clasificacion[df_clasificacion["SECTOR"].isin([1,2,3])]
df_clasificacion

## Ejercicio 15
Eliminamos a los indiduos con ingreso total = 0 (variable P47T) y reemplazamos esta variable por su logaritmo (para evitar valores muy grandes)

In [ ]:
df_clasificacion = df_clasificacion[df_clasificacion["P47T"] > 0]
df_clasificacion["P47T"] = np.log(df_clasificacion["P47T"])

## Ejercicio 16

In [ ]:
df_clasificacion_scaled = StandardScaler().set_output(transform="pandas").fit_transform(df_clasificacion)

pca = PCA(n_components=2)
componentes_principales = pca.fit_transform(df_clasificacion_scaled)

In [ ]:
(
    so.Plot(
        x=componentes_principales[:, 0],
        y=componentes_principales[:, 1],
        color = df_clasificacion["SECTOR"].astype(str)
    )
    .add(so.Dot(edgecolor="w", alpha=0.7))
    .label(x="Z1", y="Z2")
).layout(size=(8,8))

Podemos observar que los grupos estan relativamente cerca, a excepcion de una parte del sector 1 que esta un poco mas alejada o dispersa

## Ejercicio 17

In [ ]:
y = df_clasificacion["SECTOR"]
X = df_clasificacion[[c for c in df_clasificacion.columns if c != "SECTOR"]]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_train = StandardScaler().set_output(transform="pandas").fit(X_train)
X_train_scaled = scaler_train.transform(X_train)

In [ ]:
# Usamos KFold cross validation
for K in range(1, 20, 2):
    neighbor = KNeighborsClassifier(n_neighbors=K)
    kf = KFold(n_splits=5)

    scores = []
    for i, (train_index, val_index) in enumerate(kf.split(X_train_scaled)):
        neighbor.fit(X_train_scaled.iloc[train_index], y_train.iloc[train_index])
        y_pred = neighbor.predict(X_train_scaled.iloc[val_index])
        scores.append(accuracy_score(y_train.iloc[val_index], y_pred))

    print(K, sum(scores)/5)

Elegimos k = 7, ya que el score es el mayor

In [ ]:
X_test_scaled = scaler_train.transform(X_test)
neighbor_best = KNeighborsClassifier(n_neighbors=7)

neighbor_best.fit(X_train_scaled, y_train)

y_pred = neighbor_best.predict(X_test_scaled)

accuracy_score(y_pred, y_test)

Para el valor de k previamente seleccionado, la precision (porcentaje de aciertos) es de 0.92

A continuacion se proponen tres posibles variables que pueden ser valiosas para clasificar el sector de trabajo:

- NIVEL_ED: Nivel educativo
- CAT_OCUP: categoria ocupacional (patron, cuenta propia, obrero, etc)
- PP04A: estatal o privado

Otras posibilidades eran:
- CH04: Sexo
- CH06: Cuantos años cumplidos tiene
- EMPLEO
- PP04B1: Si presta servicio doméstico en hogares particulares


In [ ]:
# score con todas: 0.9212278106508875
variables_a_eliminar = ["NIVEL_ED", "CAT_OCUP", "PP04A"]
columnas_menos_precisas = []

for v in variables_a_eliminar:
  for c in X_train.columns:
    if v not in c:
      columnas_menos_precisas.append(c)
    else:
      print("Eliminando columna:", c)
print()

if columnas_menos_precisas == []:
  columnas_menos_precisas = X_train.columns

scaler_new = StandardScaler().set_output(transform="pandas").fit(X_train[columnas_menos_precisas])
X_scaled_new = scaler_new.transform(X_train[columnas_menos_precisas])

X_test_scaled = scaler_new.transform(X_test[columnas_menos_precisas])
neighbor_best = KNeighborsClassifier(n_neighbors=7)

neighbor_best.fit(X_scaled_new[columnas_menos_precisas], y_train)

y_pred_new = neighbor_best.predict(X_test_scaled[columnas_menos_precisas])

accuracy_score(y_pred_new, y_test)

# Regresion

## Ejercicio 18
df_regresion contiene solo las personas ocupadas, con ingreso
total positivo (variable P47T).

In [ ]:
df_regresion = datos.copy()

In [ ]:
df_regresion = df_regresion[df_regresion["ESTADO"] == 1] # Corresponden a personas ocupadas

In [ ]:
df_regresion = df_regresion[df_regresion["P47T"] > 0] # Ingreso total positivo

## Ejercicio 19
Aplicamos logaritmo a la variable P47T

In [ ]:
df_regresion["P47T"] = np.log(df_regresion["P47T"])

In [ ]:
df_regresion.head()

## Ejercicio 20
Proponemos tres modelos distintos:
- modelo de regresion lineal
- Ridge
- Red neuronal

Para cada uno de ellos separamos los datos en entrenamiento y testeo, entrenamos y validamos calculando el R2

Comenzamos preparando los datos

In [ ]:
df_regresion_copy = df_regresion.copy()

# Mezclamos
df_regresion_copy = df_regresion_copy.sample(len(df_regresion_copy), random_state=10)

y = df_regresion_copy.pop("P47T")

In [ ]:
# Seleccionamos todas las columnas numericas
numeric_cols = df_regresion_copy.select_dtypes(include='number').columns
numeric_cols

In [ ]:
X = df_regresion_copy[numeric_cols]

In [ ]:
# Separamos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=10)

In [ ]:
# Escalamos usando X_train, luego aplicamos el escalamiento a train y test
scaler = StandardScaler().set_output(transform="pandas").fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

### Regresion Lineal simple

In [ ]:
modelo_lineal = LinearRegression()

modelo_lineal.fit(X_train, y_train)
y_pred = modelo_lineal.predict(X_test)

print("Regresion Lineal:", root_mean_squared_error(y_test, y_pred))

### Ridge con distintos valores de alpha

In [ ]:
# Usamos KFold cross validation
for alpha in [0.1, 1, 10, 100, 500, 1000, 2000, 10000]:
    modelo_ridge = Ridge(alpha=alpha)
    kf = KFold(n_splits=5)

    scores = []
    for i, (train_index, val_index) in enumerate(kf.split(X_train)):
        modelo_ridge.fit(X_train.iloc[train_index], y_train.iloc[train_index])
        y_pred = modelo_ridge.predict(X_train.iloc[val_index])
        scores.append(r2_score(y_train.iloc[val_index], y_pred))

    print(alpha, sum(scores)/5)

El mejor score fue de 0.4133, obtenido para el hiperparametro alpha de 500

In [ ]:
alpha_best = 500

y_pred = modelo_ridge.predict(X_test)

print(f"Ridge con alpha = {alpha_best}:", root_mean_squared_error(y_test, y_pred))

### Red neuronal

In [ ]:
keras.utils.set_random_seed(11)

# Callback para monitorear el progreso
class PrintProgress(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:  # Imprime cada 10 épocas
            print(f"Época {epoch+1:>3} | loss: {logs['loss']:.4f} | val_loss: {logs['val_loss']:.4f}")

def graficar_error(history, error_name):
    x_arr = np.array(history.epoch)    # en el atributo epoch, history guarda una lista de epocas
    plot = (
        so.Plot()
        .add(so.Line(color='blue'), x=x_arr, y=history.history[error_name], label='Entrenamiento')
        .add(so.Line(color='orange'), x=x_arr, y=history.history[f'val_{error_name}'], label='Validacion')
        .label(title=error_name)
    )
    plot.show()

In [ ]:
# Paso 1: arquitectura de la red
model = keras.Sequential([
    keras.layers.Input(shape=(len(X_train.columns),)),
    keras.layers.Dense(2, activation="softmax"),
    keras.layers.Dense(1, activation="relu")
])

# Paso 2: optimizador
optimizer = keras.optimizers.SGD(learning_rate=0.01)

# Paso 3: compilamos
model.compile(
    optimizer=optimizer,
    loss='mean_squared_error',
)

# Paso 4: entrenamiento
hist = model.fit(
    X_train.to_numpy(), y_train.to_numpy(),
    epochs=50,
    batch_size=20,
    validation_split=0.2,
    verbose=0,
    callbacks=[PrintProgress()]
)

# Paso 5: evaluación en test
results = model.evaluate(
    X_test.to_numpy(), y_test.to_numpy(),
    verbose=0,
    batch_size=len(y_test),
    return_dict=True
)
print(f"\nMSE en test: {results['loss']:.4f}")

# Paso 6: gráfico
graficar_error(hist, 'loss')

In [ ]:
# Obtenemos las predicciones del modelo en test
y_pred = model.predict(X_test.to_numpy()).flatten()  # .flatten() para pasar de shape (n,1) a (n,)

# Calculamos RMSE
print(f"Red Neuronal:", root_mean_squared_error(y_test, y_pred))

In [ ]:
weights, bias = model.layers[0].get_weights()  # shape: (44, 2)

importancia = np.abs(weights).mean(axis=1)  # promedio entre las 2 neuronas

pd.Series(importancia, index=X_train.columns, name='Importancia en RN').sort_values(ascending=False)

## Ejercicio 21
El modelo Ridge depende del hiperparametro alpha. Dicho hiperparametro fue elegido previamente probando distintos valores y observando el score R2

## Ejercicio 22
A partir de los resultados observados, el mejor modelo, es decir, aquel que devuelve mejor RMSE, es el de Rdige (da mayor valor, pero como es menor que 1 y es raiz cuadrada, seria el menor)

## Ejercicio 23
Al trabajar con el modelo de red neuronal, podemos ver en la tabla de features mas relevantes que las primeras tres son:

- PP3E_TOT: Total de horas que trabajó en la semana en la ocupación principal
- PP02D: Durante esos 30 días, consultó amigos/parientes, puso carteles, hizo
algo para ponerse por su cuenta
- PP02C4: Hizo algo para ponerse por su cuenta

Aparentemente, el sueldo de una persona esta bastante influido por la cantidad de horas trabajadas, y si trabaja por su cuenta o no